# 02. Carga: dengue no Postgres

Leva para o banco as notificações baixadas no `01_coleta_dengue.ipynb` (opção B, CSV anual).

| Entrada | Saída no banco `dengue-eda` |
|---|---|
| `data/raw/dengue_csv/ano=AAAA/DENGBR{aa}.csv` | `sinan_dengue`: uma linha por notificação |
| | `sinan_dengue_carga`: registro de cada ano carregado |

**Como fica a tabela `sinan_dengue`**
- É o dado bruto: Brasil inteiro, sem filtro e sem limpeza.
- Tem todas as colunas que aparecem nos CSVs (196), como texto, mais `ano_arquivo` (o ano do arquivo de origem).
  Uma coluna que não existe num período fica vazia nele.
- É dividida por ano, então recarregar um ano não mexe nos outros.

**Como fica a tabela `sinan_dengue_carga`**
- Uma linha por ano carregado: arquivo de origem, número de linhas e data da carga.
- Serve para não carregar duas vezes: rodando de novo, os anos já carregados são pulados.

**Neste notebook**
1. Conexão
2. Carga
3. Conferência

In [1]:
import sys
from pathlib import Path
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists())
sys.path.append(str(RAIZ / "src"))

import pandas as pd
import psycopg
from carga.dengue import carga_dengue
from coleta.dengue import coleta_dengue_csv as dengue_csv
from config import ANO_INICIO, ANO_FIM_COLETA, BANCO

def consulta(q):
    with psycopg.connect(**BANCO) as con:
        cur = con.execute(q)
        return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

## 1. Conexão
Banco Postgres local. Os dados de acesso ficam no arquivo `.env` na raiz do projeto (modelo em `.env.example`).


In [2]:
consulta("select version()")

,version
0,"PostgreSQL 18.4 on x86_64-windows, compiled by..."


## 2. Carga
Carrega um ano por vez (leva poucos minutos no total). Para carregar de novo os anos já carregados, use `REFAZER = True`.

In [3]:
REFAZER = False
carga_dengue.carregar(range(ANO_INICIO, ANO_FIM_COLETA + 1), refazer=REFAZER)

ano=2000: ja carregado (172,855 linhas)


ano=2001: ja carregado (488,590 linhas)


ano=2002: ja carregado (897,093 linhas)


ano=2003: ja carregado (416,609 linhas)


ano=2004: ja carregado (136,867 linhas)


ano=2005: ja carregado (261,501 linhas)


... (17 outputs left out)

ano=2023: ja carregado (1,645,956 linhas)


ano=2024: ja carregado (6,564,924 linhas)


ano=2025: ja carregado (1,643,215 linhas)


ano=2026: ja carregado (489,197 linhas)


## 3. Conferência

### 3.1 O banco tem tudo o que estava no CSV?
Número de linhas por ano no banco e no CSV. Devem ser iguais.

In [4]:
no_banco = consulta("select ano_arquivo as ano, count(*) as banco from sinan_dengue group by 1 order by 1").set_index("ano")
no_csv = pd.Series({ano: len(pd.read_csv(dengue_csv.DESTINO / f"ano={ano}" / f"DENGBR{ano % 100:02d}.csv",
                                         usecols=["NU_ANO"], dtype=str))
                    for ano in range(ANO_INICIO, ANO_FIM_COLETA + 1)}, name="csv")
linhas = no_banco.join(no_csv)
linhas["igual"] = linhas["banco"] == linhas["csv"]
print(f"Total no banco: {linhas['banco'].sum():,} | anos iguais ao CSV: {linhas['igual'].sum()} de {len(linhas)}")
linhas

Total no banco: 32,530,748 | anos iguais ao CSV: 27 de 27


,banco,csv,igual
ano,,,
2000,172855,172855,True
2001,488590,488590,True
2002,897093,897093,True
2003,416609,416609,True
2004,136867,136867,True
2005,261501,261501,True
2006,411022,411022,True
2007,717097,717097,True
2008,919324,919324,True


### 3.2 Campos principais
% de linhas preenchidas em alguns campos importantes, por ano. Idade e classificação mudam de nome em 2007:
`nu_idade` e `con_classi` até 2006; `nu_idade_n` e `classi_fin` depois.

In [5]:
consulta("""
    select ano_arquivo as ano, count(*) as linhas,
           round(100.0 * count(id_mn_resi) / count(*)) as id_mn_resi,
           round(100.0 * count(dt_sin_pri) / count(*)) as dt_sin_pri,
           round(100.0 * count(cs_sexo)    / count(*)) as cs_sexo,
           round(100.0 * count(nu_idade)   / count(*)) as nu_idade,
           round(100.0 * count(nu_idade_n) / count(*)) as nu_idade_n,
           round(100.0 * count(con_classi) / count(*)) as con_classi,
           round(100.0 * count(classi_fin) / count(*)) as classi_fin
    from sinan_dengue group by 1 order by 1""").set_index("ano")

,linhas,id_mn_resi,dt_sin_pri,cs_sexo,nu_idade,nu_idade_n,con_classi,classi_fin
ano,,,,,,,,
2000,172855,100,100,100,100,0,89,0
2001,488590,100,100,100,99,0,90,0
2002,897093,100,100,100,99,0,84,0
2003,416609,100,100,100,100,0,89,0
2004,136867,100,100,100,100,0,94,0
2005,261501,100,100,100,100,0,92,0
2006,411022,100,100,100,100,0,90,0
2007,717097,100,100,100,0,100,0,98
2008,919324,100,100,100,0,100,0,97


### 3.3 Espaço ocupado no banco

In [6]:
consulta("""
    select pg_size_pretty(sum(pg_total_relation_size(inhrelid))) as tamanho_total,
           count(*) as particoes
    from pg_inherits where inhparent = 'sinan_dengue'::regclass""")

,tamanho_total,particoes
0,7916 MB,27


### 3.4 Exemplo: notificações de quem mora no Nordeste
Municípios de residência com código começando de 21 a 29 (o recorte do Nordeste é feito na limpeza).

In [7]:
consulta("""
    select ano_arquivo as ano, count(*) as notificacoes_ne
    from sinan_dengue where left(id_mn_resi, 2) between '21' and '29'
    group by 1 order by 1""").set_index("ano").T

ano,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
notificacoes_ne,83260,181002,326162,207681,41861,124677,107714,175517,299721,177797,...,150910,104638,298269,221930,217989,243456,105250,345534,101044,128326
